# NEXT Transformer — Cached EnergyBench Run

This notebook trains the NEXT Transformer from a validated disk-backed token cache while keeping the shared EnergyBench training and evaluation functions unchanged. Build and benchmark the cache with the standalone scripts before running this notebook.


## 1. Paths and imports

The workflow copy owns splitting, training, checkpoints, metrics, evaluation, and plotting. The NEXT package owns tokenization, cache loading, positional encoding, and the Transformer model.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import time

import pandas as pd
import torch

PROJECT_ROOT = Path(
    "/home/klz/Data/zeronu_benchmark/Transformer_Approach"
)
NEXT_ROOT = PROJECT_ROOT / "next_detector"
WORKFLOW_ROOT = PROJECT_ROOT / "evalutaions_workflow"

for path in (WORKFLOW_ROOT, NEXT_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from simple_energybench import (
    EvaluationConfig,
    TrainingConfig,
    evaluate_classification,
    set_seed,
    train_model,
)
from next_transformer import (
    NEXTTransformerClassifier,
    TokenizationConfig,
    find_token_cache,
    prepare_cached_dataset,
    validate_token_cache,
)

import simple_energybench
import next_transformer

print("EnergyBench:", simple_energybench.__file__)
print("Transformer:", next_transformer.__file__)


EnergyBench: /home/klz/Data/zeronu_benchmark/Transformer_Approach/evalutaions_workflow/simple_energybench/__init__.py
Transformer: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/next_transformer/__init__.py


## 2. Official configuration

Only `num_workers` is changed from the collaboration defaults. It is a data-loading throughput setting; model optimization and evaluation settings remain standard.


In [2]:
DATA_ROOT = Path(
    "/home/klz/Data/zeronu_benchmark/NEXT"
)
OUTPUT_ROOT = NEXT_ROOT / "results"
MANIFEST_PATH = OUTPUT_ROOT / "event_split.json"
CACHE_ROOT = NEXT_ROOT / "token_cache"
FINAL_OUTPUT_ROOT = OUTPUT_ROOT / "final_cached_v1"
FINAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

training_config = TrainingConfig(num_workers=8)
evaluation_config = EvaluationConfig()
set_seed(training_config.seed, training_config.deterministic)

print(training_config)
print(evaluation_config)
print("Dataset:", DATA_ROOT)
print("Split manifest:", MANIFEST_PATH)
print("Cache root:", CACHE_ROOT)
print("Outputs:", FINAL_OUTPUT_ROOT)


TrainingConfig(batch_size=64, epochs=50, learning_rate=0.0005, weight_decay=0.0001, gradient_clip_norm=1.0, early_stopping_patience=5, early_stopping_min_delta=0.0, seed=42, deterministic=False, use_amp=True, amp_precision='auto', optimizer='adamw', scheduler='cosine', classification_loss='bce_with_logits', regression_loss='mse', device='auto', num_workers=8)
EvaluationConfig(energy_bin_width_kev=5.0, energy_grid_min_kev=0.0, energy_grid_max_kev=3000.0, energy_grid_bin_count=600, energy_unit='MeV', matching_target='overlap', min_per_class=20, min_valid_bins=2, support_trim_quantile=0.005, energy_roi=None, min_coverage=0.5, target_tpr=0.9, score_bins=20, min_per_bin=20, distance_correlation_max_samples=1200, performance_bins=10, fractional_energy_floor=None, seed=42)
Dataset: /home/klz/Data/zeronu_benchmark/NEXT
Split manifest: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/event_split.json
Cache root: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next

## 3. Experiments and deadline gate

All four ablations run sequentially. Each tokenization cache is loaded once and reused by both positional encodings. A model is marked complete only after training and held-out test evaluation are written to the summary CSV. Re-running the notebook skips completed models.


In [3]:
ALL_EXPERIMENTS = [
    {
        "model_id": "transformer_001_sampled_hits_coordinate_mlp",
        "tokenization": "sampled_hits",
        "position_encoding": "coordinate_mlp",
    },
    {
        "model_id": "transformer_002_voxel_coordinate_mlp",
        "tokenization": "voxel",
        "position_encoding": "coordinate_mlp",
    },
    {
        "model_id": "transformer_003_voxel_fourier_xyz",
        "tokenization": "voxel",
        "position_encoding": "fourier_xyz",
    },
    {
        "model_id": "transformer_004_sampled_hits_fourier_xyz",
        "tokenization": "sampled_hits",
        "position_encoding": "fourier_xyz",
    },
]

# Run the complete 2-tokenization x 2-position-encoding benchmark.
RUN_MODEL_IDS = {
    "transformer_001_sampled_hits_coordinate_mlp",
    "transformer_002_voxel_coordinate_mlp",
    "transformer_003_voxel_fourier_xyz",
    "transformer_004_sampled_hits_fourier_xyz",
}

EXPERIMENTS = [
    experiment
    for experiment in ALL_EXPERIMENTS
    if experiment["model_id"] in RUN_MODEL_IDS
]

print("Selected experiments:")
for experiment in EXPERIMENTS:
    print(" -", experiment["model_id"])


Selected experiments:
 - transformer_001_sampled_hits_coordinate_mlp
 - transformer_002_voxel_coordinate_mlp
 - transformer_003_voxel_fourier_xyz
 - transformer_004_sampled_hits_fourier_xyz


## 4. Resolve and validate required caches

A cache is accepted only when its split hash, tokenizer configuration, tokenizer-source hash, shapes, dtypes, and `_SUCCESS` marker all validate.


In [4]:
tokenization_configs = {
    "voxel": TokenizationConfig(
        tokenization="voxel",
        max_tokens=512,
        voxel_size=15.0,
        coordinate_scale=1000.0,
        center_coordinates=True,
        voxel_truncation="occupancy",
        seed=training_config.seed,
    ),
    "sampled_hits": TokenizationConfig(
        tokenization="sampled_hits",
        max_tokens=512,
        voxel_size=15.0,
        coordinate_scale=1000.0,
        center_coordinates=True,
        voxel_truncation="occupancy",
        seed=training_config.seed,
    ),
}

required_tokenizations = {
    experiment["tokenization"]
    for experiment in EXPERIMENTS
}

data_by_tokenization = {}
cache_reports = {}
loader_optimizations = {}

for tokenization_name in sorted(required_tokenizations):
    config = tokenization_configs[tokenization_name]
    trim_padding = tokenization_name == "voxel"
    compact_training_batches = True
    cache_dir = find_token_cache(
        CACHE_ROOT,
        DATA_ROOT,
        MANIFEST_PATH,
        config,
    )
    report = validate_token_cache(
        cache_dir,
        DATA_ROOT,
        MANIFEST_PATH,
        config,
    )
    prepared = prepare_cached_dataset(
        cache_dir,
        batch_size=training_config.batch_size,
        num_workers=training_config.num_workers,
        seed=training_config.seed,
        pin_memory=torch.cuda.is_available(),
        trim_padding=trim_padding,
        compact_training_batches=compact_training_batches,
    )
    data_by_tokenization[tokenization_name] = prepared
    cache_reports[tokenization_name] = report
    loader_optimizations[tokenization_name] = {
        "trim_padding": trim_padding,
        "compact_training_batches": compact_training_batches,
    }
    print()
    print(tokenization_name, "cache:", cache_dir)
    print("counts:", prepared.counts)
    print("loader optimizations:", loader_optimizations[tokenization_name])



sampled_hits cache: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/token_cache/sampled_hits_b4460785f223_4ba5d22f5536
counts: {'boundary_files': ['0nubb_part_4/ATPC_0nubb_5bar_Efilt_5.0percent_smear_3983.h5', '0nubb_part_5/ATPC_0nubb_5bar_Efilt_5.0percent_smear_5204.h5', 'Bi_part_6/ATPC_Bi_ion_5bar_Efilt_5.0percent_smear_3505.h5', 'Bi_part_7/ATPC_Bi_ion_5bar_Efilt_5.0percent_smear_3894.h5'], 'by_class': {'0nubb': {'test': 64879, 'total': 648793, 'train': 519034, 'validation': 64880}, 'Bi214': {'test': 51670, 'total': 516696, 'train': 413357, 'validation': 51669}}, 'fractions': {'test': 0.10000008580089559, 'train': 0.7999998283982088, 'validation': 0.10000008580089559}, 'test': 116549, 'total': 1165489, 'train': 932391, 'validation': 116549}
loader optimizations: {'trim_padding': False, 'compact_training_batches': True}

voxel cache: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/token_cache/voxel_b4460785f223_ae165e692ff1
counts: {'boundary_fil

## 5. Official preflight

These assertions stop the run before training if the cache, split, standard training configuration, or CUDA environment is wrong.


In [5]:
EXPECTED_COUNTS = {
    "total": 1_165_489,
    "train": 932_391,
    "validation": 116_549,
    "test": 116_549,
}

for tokenization_name, prepared in data_by_tokenization.items():
    for name, expected in EXPECTED_COUNTS.items():
        assert prepared.counts[name] == expected, (
            tokenization_name,
            name,
            prepared.counts[name],
            expected,
        )
    assert prepared.manifest_path.resolve() == MANIFEST_PATH.resolve()
    assert prepared.cache_manifest_path.is_file()

assert training_config.epochs == 50
assert training_config.batch_size == 64
assert training_config.learning_rate == 5e-4
assert training_config.early_stopping_patience == 5
assert training_config.num_workers == 8
assert torch.cuda.is_available()
assert loader_optimizations["voxel"] == {
    "trim_padding": True,
    "compact_training_batches": True,
}
assert loader_optimizations["sampled_hits"] == {
    "trim_padding": False,
    "compact_training_batches": True,
}

print("Official cached-run preflight passed.")
print("GPU:", torch.cuda.get_device_name(0))
print("Selected models:", len(EXPERIMENTS))


Official cached-run preflight passed.
GPU: NVIDIA GeForce RTX 5090
Selected models: 4


## 6. Train and evaluate

EnergyBench saves `best_model.pt` and `last_model.pt` while training. When early stopping or epoch 50 is reached, `train_model` restores the best-validation-AUC weights in memory, and `evaluate_classification` evaluates those weights once on the held-out test split.


In [6]:
SUMMARY_PATH = FINAL_OUTPUT_ROOT / "transformer_results.csv"

if SUMMARY_PATH.is_file():
    existing_results = pd.read_csv(SUMMARY_PATH)
    experiment_rows = existing_results.to_dict(orient="records")
    completed_model_ids = set(existing_results["model_id"].astype(str))
    print("Previously completed:", sorted(completed_model_ids))
else:
    experiment_rows = []
    completed_model_ids = set()

for experiment in EXPERIMENTS:
    model_id = experiment["model_id"]
    if model_id in completed_model_ids:
        print("Skipping completed model:", model_id)
        continue

    tokenization_name = experiment["tokenization"]
    position_encoding = experiment["position_encoding"]
    prepared_data = data_by_tokenization[tokenization_name]
    representation_config = tokenization_configs[tokenization_name]
    cache_report = cache_reports[tokenization_name]
    run_root = FINAL_OUTPUT_ROOT / model_id

    print()
    print("=" * 80)
    print(model_id)
    print("Tokenization:", tokenization_name)
    print("Position encoding:", position_encoding)
    print("Cache:", cache_report["cache_dir"])
    print("Output:", run_root)
    print("=" * 80)

    set_seed(training_config.seed, training_config.deterministic)
    model = NEXTTransformerClassifier(
        position_encoding=position_encoding,
        feature_dim=2,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        num_frequencies=6,
    )
    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    print("Trainable parameters:", f"{parameter_count:,}")

    representation_path = run_root / "representation_config.json"
    representation_path.parent.mkdir(parents=True, exist_ok=True)
    representation_record = {
        **representation_config.to_dict(),
        "position_encoding": position_encoding,
        "feature_dim": 2,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.1,
        "num_frequencies": 6,
        "parameter_count": parameter_count,
        "trim_padding": loader_optimizations[tokenization_name][
            "trim_padding"
        ],
        "compact_training_batches": loader_optimizations[
            tokenization_name
        ]["compact_training_batches"],
        "manifest_path": str(prepared_data.manifest_path),
        "cache_manifest_path": str(prepared_data.cache_manifest_path),
        "source_manifest_sha256": cache_report["source_manifest_sha256"],
        "tokenization_config_sha256": cache_report[
            "tokenization_config_sha256"
        ],
        "tokenization_source_sha256": cache_report[
            "tokenization_source_sha256"
        ],
    }
    representation_path.write_text(
        json.dumps(representation_record, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    training_start = time.perf_counter()
    history = train_model(
        model,
        prepared_data.train_loader,
        prepared_data.validation_loader,
        config=training_config,
        task="classification",
        output_dir=run_root / "training",
        overwrite=False,
    )
    training_seconds = time.perf_counter() - training_start

    print("Best epoch:", history["best_epoch"])
    print("Best validation AUC:", history["best_metric"])

    evaluation_start = time.perf_counter()
    results = evaluate_classification(
        model,
        prepared_data.test_loader,
        device=training_config.device,
        output_dir=run_root / "evaluation",
        config=evaluation_config,
        overwrite=False,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start

    row = {
        "model_id": model_id,
        "tokenization": tokenization_name,
        "position_encoding": position_encoding,
        "parameter_count": parameter_count,
        "best_epoch": history["best_epoch"],
        "best_validation_auc": history["best_metric"],
        "test_events": results["n_events"],
        "inclusive_auc": results["auc"],
        "energy_matched_auc": results["matched_auc"],
        "matched_auc_status": results["matched_auc_status"],
        "common_support_auc": results["common_support_auc"],
        "shortcut_gap": results["shortcut_gap"],
        "energy_independence_score": results["energy_independence_score"],
        "worst_energy_independence_score": results[
            "worst_energy_independence_score"
        ],
        "training_seconds": training_seconds,
        "evaluation_seconds": evaluation_seconds,
        "cache_manifest_path": str(prepared_data.cache_manifest_path),
        "cache_config_sha256": cache_report["tokenization_config_sha256"],
    }
    experiment_rows.append(row)
    pd.DataFrame(experiment_rows).to_csv(SUMMARY_PATH, index=False)
    display(pd.DataFrame([row]))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Previously completed: ['transformer_001_sampled_hits_coordinate_mlp']
Skipping completed model: transformer_001_sampled_hits_coordinate_mlp

transformer_002_voxel_coordinate_mlp
Tokenization: voxel
Position encoding: coordinate_mlp
Cache: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/token_cache/voxel_b4460785f223_ae165e692ff1
Output: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/final_cached_v1/transformer_002_voxel_coordinate_mlp
Trainable parameters: 111,105


Epoch 001/050 | train loss 0.443215 | val loss 0.975452 | val auc 0.899707


Epoch 002/050 | train loss 0.368593 | val loss 0.299794 | val auc 0.944521


Epoch 003/050 | train loss 0.324155 | val loss 0.278406 | val auc 0.958665


Epoch 004/050 | train loss 0.284102 | val loss 0.45635 | val auc 0.958191


...

Epoch 046/050 | train loss 0.228475 | val loss 0.256469 | val auc 0.961909


Epoch 047/050 | train loss 0.226832 | val loss 0.26682 | val auc 0.962019


Epoch 048/050 | train loss 0.227139 | val loss 0.260405 | val auc 0.962081


Epoch 049/050 | train loss 0.225562 | val loss 0.263864 | val auc 0.962062
Early stopping; restoring best model from epoch 44.
Best epoch: 44
Best validation AUC: 0.9622048362288272


,model_id,tokenization,position_encoding,parameter_count,best_epoch,best_validation_auc,test_events,inclusive_auc,energy_matched_auc,matched_auc_status,common_support_auc,shortcut_gap,energy_independence_score,worst_energy_independence_score,training_seconds,evaluation_seconds,cache_manifest_path,cache_config_sha256
0,transformer_004_sampled_hits_fourier_xyz,sampled_hits,fourier_xyz,113409,44,0.962205,116549,0.962684,0.96273,ok,0.962672,-0.000058,0.974274,0.973252,33656.279749,24.190334,/home/klz/Data/zeronu_benchmark/Transformer_Ap...,4ba5d22f5536d1812605f13cb4d23c329a24cdcba386e9...


## 7. Results saved so far


In [7]:
if SUMMARY_PATH.is_file():
    results_dataframe = pd.read_csv(SUMMARY_PATH)
    display(
        results_dataframe.sort_values(
            "energy_matched_auc",
            ascending=False,
        ).reset_index(drop=True)
    )
else:
    print("No complete cached experiment has been evaluated yet.")


,model_id,tokenization,position_encoding,parameter_count,best_epoch,best_validation_auc,test_events,inclusive_auc,energy_matched_auc,matched_auc_status,common_support_auc,shortcut_gap,energy_independence_score,worst_energy_independence_score,training_seconds,evaluation_seconds,cache_manifest_path,cache_config_sha256
0,transformer_002_voxel_coordinate_mlp,voxel,coordinate_mlp,111105,47,0.988214,116549,0.987943,0.987718,ok,0.987933,0.000214,0.973246,0.966642,22388.583095,22.976107,/home/klz/Data/zeronu_benchmark/Transformer_Ap...,ae165e692ff169963dcfa8816ba8df26fa06140045cfe3...
1,transformer_003_voxel_fourier_xyz,voxel,fourier_xyz,113409,50,0.979266,116549,0.978971,0.978698,ok,0.978933,0.000236,0.972763,0.967611,31440.345512,54.083545,/home/klz/Data/zeronu_benchmark/Transformer_Ap...,ae165e692ff169963dcfa8816ba8df26fa06140045cfe3...
2,transformer_001_sampled_hits_coordinate_mlp,sampled_hits,coordinate_mlp,111105,23,0.975839,116549,0.975567,0.975494,ok,0.975581,0.000087,0.975270,0.971484,14092.667939,32.013669,/home/klz/Data/zeronu_benchmark/Transformer_Ap...,4ba5d22f5536d1812605f13cb4d23c329a24cdcba386e9...
3,transformer_004_sampled_hits_fourier_xyz,sampled_hits,fourier_xyz,113409,44,0.962205,116549,0.962684,0.962730,ok,0.962672,-0.000058,0.974274,0.973252,33656.279749,24.190334,/home/klz/Data/zeronu_benchmark/Transformer_Ap...,4ba5d22f5536d1812605f13cb4d23c329a24cdcba386e9...
